In [1]:
import pandas as pd
import numpy  as np
import torch

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU is enabled. Using {torch.cuda.get_device_name(0)}.")
else:
    print("GPU is not enabled. Using CPU.")

GPU is enabled. Using Tesla P100-PCIE-16GB.


In [19]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

# Replace with the actual model name or local path
model_name = "thegoodfellas/tgf-xlm-roberta-base-pt-br"  # Example, check Hugging Face or GitHub
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at thegoodfellas/tgf-xlm-roberta-base-pt-br and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
import re
from sklearn.model_selection import train_test_split

# Load the training data
train_df = pd.read_csv('/kaggle/input/tupy-e/binary_train.csv')
train_df = train_df[['text', 'hate','aggressive']].dropna()

# Load the test data
test_df = pd.read_csv('/kaggle/input/tupy-e/binary_test.csv')
test_df = test_df[['text', 'hate','aggressive']].dropna()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df['text'] = train_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)


In [21]:
label = np.zeros((len(train_df['text']), 2))
for i in range(len(train_df)):
    if train_df['hate'][i] == 0:
        label[i] = [1, 0]
    else:
        label[i] = [0, 1]


In [22]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    label.tolist(),
    test_size=0.2,
    random_state=42
)

# For training and validation
from datasets import load_dataset, Dataset
train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels}) # Changed 'hate' to 'labels'
val_dataset = Dataset.from_dict({'text': val_texts, 'labels': val_labels}) # Changed 'hate' to 'labels'

# For testing
test_labels = [[1, 0] if label == 0 else [0, 1] for label in test_df['hate'].tolist()]
test_dataset = Dataset.from_dict({'text': test_df['text'].tolist(), 'labels': test_labels}) # Changed 'hate' to 'labels'

# Tokenize all
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/27947 [00:00<?, ? examples/s]

Map:   0%|          | 0/6987 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',  # Directory to save the model checkpoints
    eval_strategy="epoch",  # Evaluate after each epoch
    
    logging_strategy="no",  # Disables logging
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=10,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)



In [24]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Check if labels are multilabel-indicator (2D array)
    if labels.ndim == 2 and labels.shape[1] > 1:
        # If multilabel-indicator, convert to binary format
        labels = labels.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.273870,0.886647,0.506234,0.525226,0.488568
2,No log,0.237726,0.901388,0.525809,0.614148,0.459687
3,No log,0.295387,0.884643,0.554204,0.512794,0.602888
4,No log,0.338113,0.887648,0.560224,0.524109,0.601685
5,No log,0.443357,0.894375,0.543317,0.559236,0.528279
6,No log,0.486000,0.898526,0.514716,0.596825,0.452467
7,No log,0.522445,0.896379,0.539440,0.572200,0.510229
8,No log,0.546577,0.898526,0.528904,0.590504,0.478941
9,No log,0.630394,0.893230,0.523627,0.557823,0.493381
10,No log,0.651086,0.896522,0.513131,0.582569,0.458484


TrainOutput(global_step=17470, training_loss=0.14295905145427876, metrics={'train_runtime': 4481.3504, 'train_samples_per_second': 62.363, 'train_steps_per_second': 3.898, 'total_flos': 1.83829116603648e+16, 'train_loss': 0.14295905145427876, 'epoch': 10.0})

In [31]:
from sklearn.metrics import classification_report
import numpy as np
import torch

# Wrap original dataset
class FloatLabelWrapper(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):
        item = self.dataset[idx]
        item['labels'] = item['labels'].float()  # Ensure float32 labels
        return item

    def __len__(self):
        return len(self.dataset)

# Wrap your test dataset
test_dataset = FloatLabelWrapper(test_dataset)

# Predict on test set
predictions = trainer.predict(test_dataset)
logits = predictions.predictions
probs = torch.sigmoid(torch.tensor(logits)).numpy()
pred_labels = (probs >= 0.5).astype(int)
true_labels=predictions.label_ids

print("Accuracy:", accuracy_score(true_labels, pred_labels))


Accuracy: 0.9008472635676666


In [34]:
trainer.save_model("./my-roberta-hate-base-model")
tokenizer.save_pretrained("./my-roberta-hate-base-model")


('./my-roberta-hate-base-model/tokenizer_config.json',
 './my-roberta-hate-base-model/special_tokens_map.json',
 './my-roberta-hate-base-model/tokenizer.json')

In [35]:
import shutil

# Zip the saved model and tokenizer directory
shutil.make_archive('/kaggle/working/my-roberta-hate-base-model', 'zip', './my-roberta-hate-base-model')


'/kaggle/working/my-roberta-hate-base-model.zip'